# Mean-Variance Portfolio Selection Mean-Field Control Benchmark

References: `files/reference/continuous_benchmarks.tex`, Sec. "Mean-Variance Portfolio Selection" (the model and its closed forms), and `files/reference/continuous_state_space(2).tex` (the continuous-state theory the compared algorithm implements). Like `lq.ipynb`, **the perturbed objective $J^\lambda(\theta)$, its exact policy gradient, and the unperturbed optimal policy are all available in closed form** — here even though the terminal wealth law is generally non-Gaussian (only its first two moments are used, and they close exactly).

**Model.** Wealth $X_t\in\mathbb R$, monetary amount $\alpha_t\in\mathbb R$ invested in the risky asset (the rest sits in the risk-free asset). $X_{t+1}=s_tX_t+R_{t+1}\alpha_t$, with $R_{t+1}$ the risky asset's excess return (independent across $t$; only its first two moments $\bar r_t,\sigma_{R,t}^2$ matter for the exact objective/gradient — the return *law* itself is a runtime choice, see the robustness section below). There is **no running reward**; the terminal reward is $g(x,m)=x-\chi(x-\bar m)^2$, so
$$J^0(\theta) = \mathbb E[X_T] - \chi\,\mathrm{Var}(X_T)$$
the precommitment mean-variance criterion — a **reward to maximize** (`Portfolio.MAXIMIZE` is `True`), unlike `lq.ipynb`'s cost-to-minimize convention.

**Policy.** $\pi_t^\theta(\cdot\mid x,m)=\mathcal N(k_t(x-\bar m)+\ell_t,\tau_t^2)$, $\theta=((k_t,\ell_t))_{t=0}^{T-1}$ — the same genuinely time-indexed $(T,2)$ parametrization as LQ, so a trained $\theta$ cannot be evaluated at another horizon.

**The perturbation** is the same transport randomization of the population law as in `lq.ipynb`: $M_t^{\lambda,\theta}=(\mathrm{Id}+\lambda f_t)\sharp\mu_t^\theta$ with $f_t(x)=\zeta_tx+\beta_t$, $(\zeta_t,\beta_t)\sim\mathcal N(0,\rho^2)^{\otimes2}$, drawn afresh along every trajectory. Only the mean of the wealth law enters the model, so the law chart is one-dimensional with coordinate $c_t^\theta=\mu_t^\theta$.

**Baseline parameters** (reference "Training and evaluation", used directly as `PortfolioConfig`'s defaults — unlike LQ, this benchmark's reference gives a full numeric baseline case, not just formulas): $T=10$, $X_0\sim\mathcal N(1,0.04)$, $s_t=1$, $\bar r_t=0.02$, $\sigma_{R,t}=0.08$, $\chi=10$, $\tau_t=0.02$, $\rho=1$. $\theta$ is initialized at $k_t=\ell_t=0$ (the reference's own choice).

**Two model-free algorithms are compared** at equal budget, exactly as in `lq.ipynb`: **`simplex`** (`mfc.algorithms.continuous_simplex`, the continuous-state simplex-perturbed MF-REINFORCE estimator) and **`reinforce`** (`mfc.algorithms.continuous_reinforce`, the same estimator with the population-perturbation score dropped). Simplex is trained once per $\lambda$; reinforce has no perturbation scale — the randomization exists only to expose the mean-field sensitivity through a likelihood ratio — so it is trained once, on the nominal process. The closed-form `exact_gradient` is the oracle, not a competitor.

**This is the hard benchmark of the two, for a structural reason.** The reference's own $\tau=0.02$ makes every score-function estimator's magnitude proportional to $1/\tau^2=2500$ while the gradient itself is $O(10^{-2})$, so both model-free algorithms run at a very low signal-to-noise ratio and settle on a stochastic-gradient noise floor short of $\theta^\star$ rather than converging to it. That is the MSE-vs-budget trade-off of Theorem "Bias and MSE of the gradient estimator" made visible, and it is reported here rather than tuned away (see `configs/portfolio.py`'s module docstring).

In [ ]:
import sys
import time
from pathlib import Path

_notebook_start = time.perf_counter()

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
for path in (SRC, ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import pandas as pd
import torch

torch.set_default_dtype(torch.float64)
torch.set_default_device("cpu")  # small scalar/(T,B) ops: GPU kernel-launch overhead dominates
                                 # (same finding as lq.ipynb and this repo's twostate profiling)

from configs.portfolio import MAIN, MID
from mfc.environments.portfolio import Portfolio, PortfolioConfig
from mfc.plotting import diagnostics as viz
from mfc.plotting.style import apply_style, color_for, new_figure, set_style, style_legend
from scripts.train import run_continuous
from scripts.test import (
    continuous_generalization_eval,
    continuous_gradient_diagnostics,
    continuous_mean_field_term,
    continuous_objective_gap,
    continuous_oracle_gradient_estimate,
    continuous_perturbation_coverage,
    continuous_sensitivity_error,
    continuous_state_marginal_stability,
    exact_coordinate_sensitivity,
    load_runs,
)

set_style()

REF_LAM = 0.1  # the reference perturbation scale used wherever a single lambda has to be picked
LABELS = None  # per-component theta labels, set once the horizon is known

# reinforce has no perturbation scale: the randomization exists only to expose
# the mean-field sensitivity through a likelihood ratio, and reinforce drops
# that term -- so it is trained once, on the nominal process, and its runs
# carry lam=None (scripts/train.py's ALGORITHMS_WITH_PERTURBATION_SCALE).
lam_of = lambda alg: REF_LAM if alg == "simplex" else None

## Configuration and budget

This notebook demonstrates the **mid** run tier from `configs/portfolio.py`: one seed, the reference horizon $T=10$, and the full training length. Run `scripts/train.py --env portfolio --alg simplex --config main` (and `--alg reinforce`) separately for the full main-tier sweep (5 seeds, horizons $T\in\{5,10,20\}$, used by the horizon-scaling cell below when available).

**Equal budget.** Simplex spends one auxiliary batch plus one main batch per gradient step, $(n_\mathrm{aux}+B)\,T$ simulated transitions; reinforce has no auxiliary batch, so it gets the whole allocation as its main batch. The job *counts* differ, though: simplex is trained once per $\lambda$, reinforce once in total.

In [ ]:
cfg = MID
env = Portfolio(device="cpu")  # matches the default device set above and scripts.train.run_continuous's own choice
T = cfg.horizons[0]

print(f"algorithms:   {cfg.algorithms}")
print(f"lambdas:      {cfg.lambdas}")
print(f"T={T}, seeds={cfg.seeds}, lr={cfg.lr}, n_train={cfg.n_train}, baseline={cfg.baseline!r}")
print(f"budget/step:  simplex   n_aux={cfg.n_aux} + B={cfg.B} -> {cfg.transitions_per_step(T)} transitions")
print(f"              reinforce B={cfg.reinforce_B_equal_budget()}         -> {cfg.reinforce_B_equal_budget() * T} transitions (no lambda: trained on the nominal process)")
print(f"model: s={env.config.s}, r_bar={env.config.r_bar}, sigma_R={env.config.sigma_R}, chi={env.config.chi}")
print(f"tau={env.config.tau}, rho={env.config.rho}, mu0={env.config.mu0}, Sigma0={env.config.Sigma0}")
print(f"returns: {env.config.return_distribution}")

LABELS = [f"{name}_{t}" for t in range(T) for name in ("k", "l")]

## Ground truth: the optimal policy

The unperturbed ($\lambda=0$) optimal $\theta^\star$ (`Portfolio.optimal_theta`: $k_t^\star=-s_t\bar r_t/h_t$ constant, $\ell_t^\star$ decaying geometrically away from the terminal time) and its value $J^0(\theta^\star)$ — exact, closed-form. Every plot below compares the learned policies against this, and against $J^0(0)$, the value of not investing at all.

In [ ]:
theta_star = env.optimal_theta(T)
J_star = env.exact_objective(theta_star, 0.0).item()
J_zero = env.exact_objective(torch.zeros(T, 2), 0.0).item()
grad_at_star = env.exact_gradient(theta_star, 0.0)

print("theta* (k_t, l_t):")
print(theta_star)
print(f"J^0(theta*)      = {J_star:.6f}")
print(f"J^0(theta=0)     = {J_zero:.6f}   (never invest: terminal wealth is just X_0)")
print(f"||grad J^0(theta*)|| = {grad_at_star.norm().item():.2e}  (should be ~0: theta* is a stationary point)")

mu_star, Sigma_star = env.forward_moments(theta_star, 0.0)
print(f"mu_t under theta*: {[round(v, 4) for v in mu_star.tolist()]}")

## Train (or load cached results)

Loads every saved run under `runs/portfolio/mid/` for both algorithms; trains first if none exist yet. `J/J*` is reported against the *achievable range*: `(J - J^0(0)) / (J^0(theta*) - J^0(0))` is the fraction of the available improvement over doing nothing that the learned policy actually captures, which is the informative number here (raw $J/J^\star$ compresses everything into $[0.75,1]$).

In [ ]:
runs_dir = ROOT / "runs" / "portfolio" / cfg.name

runs = []
for alg in cfg.algorithms:
    if not list(runs_dir.glob(f"{alg}_*_seed*.pt")):
        run_continuous("portfolio", alg, cfg.name)
    runs += load_runs("portfolio", alg, cfg.name, device="cpu")

by_alg_lambda = {(r["alg"], r["lam"]): r for r in runs}
theta_simplex = by_alg_lambda[("simplex", REF_LAM)]["theta_final"]
theta_reinforce = by_alg_lambda[("reinforce", None)]["theta_final"]

captured = lambda J: (J - J_zero) / (J_star - J_zero)
total_train_seconds = sum(r["elapsed_seconds"] for r in runs)
print(f"{len(runs)} runs loaded; total training compute time: {total_train_seconds:.1f}s ({total_train_seconds / 60:.1f} min)")
pd.DataFrame(
    [
        {"alg": r["alg"], "lambda": r["lam"] if r["lam"] is not None else "-", "seed": r["seed"],
         "elapsed_s": round(r["elapsed_seconds"], 1),
         "final J^0": r["validation_J"][-1].item(), "captured": captured(r["validation_J"][-1].item())}
        for r in sorted(runs, key=lambda r: (r["alg"], r["lam"] if r["lam"] is not None else -1.0, r["seed"]))
    ]
).set_index(["alg", "lambda", "seed"])

## Evolution of the validation objective

The exact validation objective $J^0(\theta_m)$ (closed-form, not Monte Carlo) every `validate_every` training iterations — one line per $\lambda$ for simplex and a single line for reinforce, which has none — against the optimal reference line. Both curves rise quickly out of $J^0(0)=0.600$ and then oscillate on the noise floor described in the header — the plateau, not the final point, is what the comparison is about.

In [ ]:
fig, ax = viz.plot_validation_curve(runs, optimal_J=J_star)
ax.axhline(J_zero, color="black", linestyle=":", linewidth=1.5, label="J^0(0) (never invest)")
style_legend(ax)
ax.set_title("Validation reward by training iteration")

## Learned theta vs optimal theta

$k_t$ (the deviation gain, which controls the variance term) and $\ell_t$ (the level, which controls the mean) across $t$ for each algorithm's $\lambda=0.1$ policy, against $\theta^\star$ (dashed). `plot_lq_theta`'s two components are $(k_t,\ell_t)$ here.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
viz.plot_lq_theta(theta_simplex, optimal_theta=theta_star, ax=axes[0])
axes[0].set_title(f"Simplex policy parameters (lambda={REF_LAM})")
viz.plot_lq_theta(theta_reinforce, optimal_theta=theta_star, ax=axes[1])
axes[1].set_title("REINFORCE policy parameters (no perturbation)")
fig.tight_layout()

pd.DataFrame(
    [
        {"alg": alg, "||theta - theta*||": (th - theta_star).norm().item(),
         "||k - k*||": (th[:, 0] - theta_star[:, 0]).norm().item(),
         "||l - l*||": (th[:, 1] - theta_star[:, 1]).norm().item()}
        for alg, th in (("simplex", theta_simplex), ("reinforce", theta_reinforce))
    ]
).set_index("alg")

## Wealth distribution over time: learned vs optimal

$\mu_t^\theta\pm\sigma_t^\theta$ (mean $\pm$ 1 std of the wealth law) under each learned policy, against the optimal trajectory. Note that unlike LQ these are the first two moments of a law that is *not* Gaussian in general — but they are exactly what the objective depends on, and `Portfolio.forward_moments` propagates them exactly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (alg, th) in zip(axes, (("simplex", theta_simplex), ("reinforce", theta_reinforce))):
    mu_l, Sigma_l = env.forward_moments(th, 0.0)
    viz.plot_gaussian_flow(mu_l, Sigma_l, optimal_mu=mu_star, optimal_Sigma=Sigma_star, label=f"learned ({alg})", ax=ax)
    ax.set_ylabel("X_t (wealth, mean ± 1 std)")
    ax.set_title(f"Wealth distribution: {alg} vs optimal")
fig.tight_layout()

## $J^\lambda$ vs $J^0$, at the learned and optimal theta

Both sides are exact closed-form evaluations (`Portfolio.exact_objective`). The perturbation costs reward here through the variance term: $J^\lambda=\mu_T-\chi[\Sigma_T+\lambda^2\rho^2(\mu_T^2+1)]$, so the gap is exactly $O(\lambda^2)$ — `tests/test_portfolio.py` checks that the ratio between consecutive grid points is exactly 4.

In [ ]:
lambdas = list(cfg.lambdas)
J_at_learned = {lam: env.exact_objective(by_alg_lambda[("simplex", lam)]["theta_final"], lam).item() for lam in lambdas}
J_at_optimal = {lam: env.exact_objective(theta_star, lam).item() for lam in lambdas}

fig, ax = new_figure()
ax.plot(lambdas, list(J_at_learned.values()), color=color_for(0), linewidth=2, marker="o", markersize=6, label="J^lambda(theta_learned)")
ax.plot(lambdas, list(J_at_optimal.values()), color=color_for(1), linewidth=2, marker="s", markersize=6, label="J^lambda(theta*)")
ax.axhline(J_star, color="black", linestyle="--", linewidth=1.5, label="J^0(theta*)")
apply_style(ax, xlabel="perturbation scale λ", ylabel="J^lambda")
style_legend(ax)
ax.set_title("Perturbed objective across lambda values")

## Gradient bias, variance and MSE: simplex vs reinforce

The headline benchmark, at the **same fixed** $\theta^\star$ for both estimators (where $\nabla_\theta J^0=0$ exactly, so any systematic offset is pure bias). Simplex is swept over $\lambda$; reinforce has no $\lambda$, so it appears once, as the estimator it actually trains on ($\lambda=0$, the nominal process). Because both oracles are closed-form, the bias splits exactly into the $O(\lambda^2)$ perturbation term (III) and the estimation term (II) — for simplex the $O(1/n)$ plug-in error of the shared sensitivity flow, for reinforce the entire missing mean-field term.

In [ ]:
reps = 60
grad_diag = {
    "simplex": {
        lam: continuous_gradient_diagnostics(
            env, theta_star, lam=lam, B=cfg.B, n_aux=cfg.n_aux, reps=reps, algorithm="simplex",
            baseline=cfg.baseline, generator=torch.Generator(device="cpu").manual_seed(0),
        )
        for lam in lambdas
    },
    # reinforce takes no lambda: this is the estimator it actually trains on
    "reinforce": {
        0.0: continuous_gradient_diagnostics(
            env, theta_star, lam=0.0, B=cfg.reinforce_B_equal_budget(), reps=reps, algorithm="reinforce",
            baseline=cfg.baseline, generator=torch.Generator(device="cpu").manual_seed(0),
        )
    },
}

rows = []
for alg, per_lam in grad_diag.items():
    for lam, d in per_lam.items():
        rows.append({
            "alg": alg, "λ": lam,
            "||bias||": d["bias"].norm().item(), "bias SE": d["bias_se"].norm().item(),
            "bias/SE": d["bias"].norm().item() / d["bias_se"].norm().item(),
            "(III) perturbation": d["perturbation_bias"].norm().item(),
            "(II) estimation": d["estimation_bias"].norm().item(),
            "||std||": d["std"].norm().item(), "MSE": d["mse"].sum().item(),
        })
pd.DataFrame(rows).set_index(["alg", "λ"])

**bias/SE** below ~2 means the estimator is statistically indistinguishable from unbiased at this number of replications; well above 2 means a real bias — and on this benchmark neither estimator's bias is resolvable this way at all, which is exactly what the paired measurement in the next section is for. Note how much larger `||std||` is than either bias here compared to LQ — that ratio is precisely why this benchmark's training plateaus: at the training budget the Monte Carlo term dominates the estimator's MSE at every $\lambda$ in the reference's grid.

In [ ]:
fig, ax = new_figure()
ax.plot(lambdas, [grad_diag["simplex"][lam]["bias"].norm().item() for lam in lambdas],
        color=color_for(0), linewidth=2, marker="o", markersize=6, label="simplex ||bias||")
ax.plot(lambdas, [grad_diag["simplex"][lam]["bias_se"].norm().item() for lam in lambdas],
        color=color_for(0), linewidth=1.5, linestyle=":", label="simplex bias SE")
reinforce_bias = grad_diag["reinforce"][0.0]
ax.axhline(reinforce_bias["bias"].norm().item(), color=color_for(1), linewidth=2, label="reinforce ||bias|| (no lambda)")
ax.axhline(reinforce_bias["bias_se"].norm().item(), color=color_for(1), linewidth=1.5, linestyle=":", label="reinforce bias SE")
ax.set_yscale("log")
apply_style(ax, xlabel="perturbation scale λ", ylabel="gradient-bias norm vs ∇J^0(θ*)")
style_legend(ax)
ax.set_title("Gradient-estimator bias at the optimum")

## The missing mean-field term, isolated

At $\theta^\star$ the true gradient is exactly zero, so any systematic offset an estimator shows there is pure bias. Measuring that bias by independent replication is statistically wasteful, though: the two estimators share their (large) policy-score variance, so most of the replications go into re-measuring noise they have in common.

`continuous_mean_field_term` instead evaluates both on the **same rollouts** and differences them. What survives is exactly the term REINFORCE omits,
$$\sum_{t=0}^T S_t^{\lambda,\theta}(M_t)\,G_t = \sum_{t=0}^T D_t^\theta h_t\, G_t,$$
with the shared policy-score contribution cancelling sample for sample. Since $\mathbb E[\hat g_\text{simplex}]=\nabla J^\lambda$ when the sensitivity flow is exact, this paired quantity *is* REINFORCE's bias, up to the $O(\lambda^2)$ perturbation term — measured to a fraction of the standard error the direct comparison achieves at the same cost.

The measurement is taken at the reference $\lambda$ because the omitted term only *exists* on a perturbed rollout — it is the likelihood-ratio score of the randomization itself. What it predicts, though, is the bias of the $\lambda=0$ REINFORCE that is actually trained: the two differ by the same $O(\lambda^2)$ that separates $\nabla J^\lambda$ from $\nabla J^0$. The last two columns are that unperturbed estimator, measured directly and far more noisily.

This benchmark is where the paired measurement earns its keep. The population coordinate here is only weakly sensitive to $\theta$ ($\mu_{t+1}=s\mu_t+\bar r\ell_t$ with $\bar r=0.02$), so the omitted term is *small in absolute terms* — and yet, as the printout shows, it is a large fraction of the entire perturbed gradient at the optimum. Independent replication cannot see it at all at any affordable budget: the estimators' own standard errors are two orders of magnitude larger than the quantity being measured.

In [ ]:
mf = continuous_mean_field_term(env, theta_star, lam=REF_LAM, B=cfg.B, reps=200,
                                baseline=cfg.baseline, generator=torch.Generator(device="cpu").manual_seed(0))
direct = grad_diag["reinforce"][0.0]

row = lambda label, text: print(f"{label:<44}: {text}")
row(f"omitted mean-field term (paired, lambda={REF_LAM})", f"{mf['mean'].norm().item():.4f} +- {mf['se'].norm().item():.4f}")
row("reinforce's own bias (direct, lambda=0)",
    f"{direct['bias'].norm().item():.4f} +- {direct['bias_se'].norm().item():.4f}"
    f"  <- same quantity up to O(lambda^2), {direct['bias_se'].norm().item() / mf['se'].norm().item():.0f}x noisier")
row("||grad J^lambda(theta*)||",
    f"{mf['perturbed_gradient'].norm().item():.6f}  (near zero: theta* is the optimum of J^0, "
    "and grad J^lambda - grad J^0 is only O(lambda^2))")
print("-> REINFORCE does not see a near-zero gradient at theta*. What it sees there is essentially this omitted "
      "term, which is exactly what moves the stationary point its recursion converges to away from theta*.")

pd.DataFrame(
    {
        "exact grad J^0(theta*)": grad_at_star.flatten().tolist(),
        "exact grad J^lambda(theta*)": mf["perturbed_gradient"].flatten().tolist(),
        "omitted mean-field term": mf["mean"].flatten().tolist(),
        "term SE": mf["se"].flatten().tolist(),
        "reinforce mean estimate": direct["mean_estimate"].flatten().tolist(),
        "reinforce SE": direct["bias_se"].flatten().tolist(),
    },
    index=LABELS,
)

## Robustness to non-Gaussian returns

The reference's own robustness experiment: replace the Gaussian return law with a centered, rescaled Student-$t(5)$ keeping $\bar r_t$ and $\sigma_{R,t}^2$ unchanged (`PortfolioConfig(return_distribution="student_t")`). The closed forms depend on the return law only through those two moments, so $J^\lambda$, $\nabla J^\lambda$ and $\theta^\star$ are *unchanged*; what can change is the estimator's finite-sample behaviour, since the heavier tails feed into every sampled return. The table compares the gradient estimator's bias and standard deviation under both return laws at the same $\theta^\star$.

In [ ]:
env_t = Portfolio(PortfolioConfig(return_distribution="student_t"), device="cpu")
print(f"theta* unchanged: {torch.allclose(env_t.optimal_theta(T), theta_star)}")
print(f"J^0(theta*) unchanged: {env_t.exact_objective(theta_star, 0.0).item():.6f} vs {J_star:.6f}")

rows = []
for name, e in (("gaussian", env), ("student_t", env_t)):
    d = continuous_gradient_diagnostics(
        e, theta_star, lam=REF_LAM, B=cfg.B, n_aux=cfg.n_aux, reps=reps,
        baseline=cfg.baseline, generator=torch.Generator(device="cpu").manual_seed(0),
    )
    rows.append({"returns": name, "||bias||": d["bias"].norm().item(), "bias SE": d["bias_se"].norm().item(),
                 "||std||": d["std"].norm().item(), "MSE": d["mse"].sum().item()})
pd.DataFrame(rows).set_index("returns")

## Horizon scaling

Final validation objective (as the fraction of the achievable improvement captured) and $\|\theta_\text{learned}-\theta^\star\|$ vs horizon $T$, using `runs/portfolio/main/` if it has been populated (`scripts/train.py --env portfolio --alg <alg> --config main`); otherwise this cell notes that main-tier data is needed and skips. The estimators' *simulation* cost is linear in $T$ (Remark "Linear per-iteration complexity"), but their statistical constants are not — Remark "Horizon dependence" is explicit that the stability factors may grow with the horizon.

In [ ]:
main_runs_dir = ROOT / "runs" / "portfolio" / "main"
probe = lambda alg: f"{alg}_T{MAIN.horizons[0]}" + (f"_lam{REF_LAM}" if lam_of(alg) is not None else "") + "_seed0.pt"
main_runs = [r for alg in cfg.algorithms if (main_runs_dir / probe(alg)).exists()
             for r in load_runs("portfolio", alg, "main", device="cpu")] if main_runs_dir.exists() else []

if not main_runs:
    print("no runs/portfolio/main/ data yet -- run scripts/train.py --env portfolio --alg simplex --config main "
          "(and --alg reinforce) to populate this cell")
else:
    rows = []
    fig, ax = new_figure()
    for i, alg in enumerate(cfg.algorithms):
        captured_by_T = {}
        for T_h in MAIN.horizons:
            group = [r for r in main_runs if r["alg"] == alg and r["lam"] == lam_of(alg) and r["T"] == T_h]
            th_star_h = env.optimal_theta(T_h)
            J_star_h = env.exact_objective(th_star_h, 0.0).item()
            J_zero_h = env.exact_objective(torch.zeros(T_h, 2), 0.0).item()
            J_h = sum(r["validation_J"][-1].item() for r in group) / len(group)
            captured_by_T[T_h] = (J_h - J_zero_h) / (J_star_h - J_zero_h)
            err = sum((r["theta_final"] - th_star_h).norm().item() for r in group) / len(group)
            rows.append({"alg": alg, "T": T_h, "final J^0": J_h, "captured": captured_by_T[T_h], "||theta-theta*||": err, "n_seeds": len(group)})
        viz.plot_horizon_scaling(captured_by_T, ylabel="fraction of achievable J captured", label=alg, color_index=i, ax=ax)
    ax.set_title("Horizon scaling of captured reward improvement")
    display(pd.DataFrame(rows).set_index(["alg", "T"]))

## Generalization without retraining

Evaluating the $\lambda=0.1$ simplex-learned $\theta$ exactly (no retraining, no Monte Carlo) under a different initial wealth law, different market parameters, a different variance aversion, and a stronger perturbation intensity. $\theta$ is horizon-specific, so there is no "different $T$" scenario (see `mfc.environments.lq`'s module docstring).

In [ ]:
scenarios = [
    {"name": "baseline"},
    {"name": "mu0 x 2 (richer population)", "env": Portfolio(PortfolioConfig(mu0=env.config.mu0 * 2), device="cpu")},
    {"name": "Sigma0 x 4 (more dispersed)", "env": Portfolio(PortfolioConfig(Sigma0=env.config.Sigma0 * 4), device="cpu")},
    {"name": "sigma_R x 1.5 (riskier asset)", "env": Portfolio(PortfolioConfig(sigma_R=env.config.sigma_R * 1.5), device="cpu")},
    {"name": "r_bar x 0.5 (worse edge)", "env": Portfolio(PortfolioConfig(r_bar=env.config.r_bar * 0.5), device="cpu")},
    {"name": "chi x 2 (stronger variance aversion)", "env": Portfolio(PortfolioConfig(chi=env.config.chi * 2), device="cpu")},
    {"name": "student-t returns", "env": env_t},
    {"name": "rho x 2 (stronger perturbation)", "env": Portfolio(PortfolioConfig(rho=env.config.rho * 2), device="cpu")},
]
gen_results = continuous_generalization_eval(env, theta_simplex, scenarios, lam=REF_LAM)
fig, ax = viz.plot_generalization(gen_results)
ax.set_title(f"Out-of-distribution objective with fixed learned policy")

## Sample wealth trajectories: learned vs optimal

One sampled $X_t$ trajectory under the $\lambda=0.1$ simplex-learned policy and one under $\theta^\star$, both from $X_0\sim\mathcal N(\mu_0,\Sigma_0)$ and both unperturbed ($\lambda=0$).

In [ ]:
learned_traj = env.rollout(theta_simplex, lam=0.0, B=1, generator=torch.Generator(device="cpu").manual_seed(0))["X"][:, 0]
optimal_traj = env.rollout(theta_star, lam=0.0, B=1, generator=torch.Generator(device="cpu").manual_seed(1))["X"][:, 0]

fig, ax = viz.plot_trajectories(learned_traj, optimal_traj)
ax.set_ylabel("X_t (wealth)")
ax.set_title("Sample wealth trajectories: learned vs optimal")

## Additional statistical validation (continuous-state theory)

The sections above check the estimators against each other and against training outcomes. The sections below check the theory's own claims from `files/reference/continuous_state_space(2).tex` directly, at the fixed $\theta^\star$ (where $\nabla_\theta J^0=0$ exactly) with $\lambda$ or the batch size as the only thing varying. These mirror `lq.ipynb`'s validation section one for one, so the two benchmarks can be read side by side.

### Convergence of the perturbed objective: $|J^\lambda(\theta)-J^0(\theta)|=O(\lambda^2)$

Proposition "Quantitative perturbation bounds" gives $O(\lambda)$ in general; Remark "Second-order bias" improves it to $O(\lambda^2)$ for a centered perturbation, which this one is. Here the identity is available in closed form — the whole $\lambda$-dependence of $J^\lambda$ is the $\lambda^2\rho^2(\mu_T^2+1)$ term in the terminal variance — so `|gap|/lambda^2` must be constant to machine precision. The Monte Carlo column checks the simulator against those closed forms.

In [ ]:
gaps = {lam: continuous_objective_gap(env, theta_star, lam=lam, n_samples=200_000,
                                      generator=torch.Generator(device="cpu").manual_seed(0)) for lam in lambdas}
rows = []
for lam in lambdas:
    g = gaps[lam]
    rows.append({
        "λ": lam, "J^0": g["J"].item(), "J^lambda (exact)": g["J_lambda"].item(),
        "J^lambda (MC)": g["J_lambda_mean"].item(), "MC SE": g["J_lambda_se"].item(),
        "|gap|": abs(g["gap"].item()), "|gap|/lambda^2": abs(g["gap"].item()) / lam**2,
    })
pd.DataFrame(rows).set_index("λ")

### Gradient-level convergence: $\|\nabla J^\lambda(\theta)-\nabla J^0(\theta)\|=O(\lambda^2)$

The same second-order statement at the gradient level — term **(III)** of the bias decomposition, needing no estimation since both gradients are closed-form. The oracle-sensitivity column confirms it empirically: `continuous_oracle_gradient_estimate` feeds the estimator the exact $D_t^\theta$, so its mean is $\nabla J^\lambda$ exactly.

In [ ]:
reps_oracle = 40
rows = []
for lam in lambdas:
    generator = torch.Generator(device="cpu").manual_seed(0)
    samples = torch.stack([
        continuous_oracle_gradient_estimate(env, theta_star, lam=lam, B=cfg.B, baseline=cfg.baseline, generator=generator)
        for _ in range(reps_oracle)
    ])
    exact_gap = (env.exact_gradient(theta_star, lam) - grad_at_star).norm().item()
    oracle_gap = (samples.mean(dim=0) - grad_at_star).norm().item()
    rows.append({
        "λ": lam, "||grad J^lambda - grad J^0|| (exact)": exact_gap, "exact/lambda^2": exact_gap / lam**2,
        "oracle-D estimate": oracle_gap, "SE": (samples.std(dim=0) / reps_oracle**0.5).norm().item(),
    })
pd.DataFrame(rows).set_index("λ")

### Coordinate-sensitivity estimator: $\hat D_t\to D_t^\theta=\nabla_\theta\mu_t^\theta$

`continuous_simplex.estimate_sensitivity_flow`'s single-batch forward estimator against the exact $D_t^\theta$ (autograd through `Portfolio.forward_moments`). The smoothing remainder shrinks with $\eta$; the reusable-batch Monte Carlo variance shrinks as $1/n$. $D_0=0$ exactly, since $\mu_0$ does not depend on $\theta$. Note that on this benchmark the wealth mean depends on $\theta$ only through the $\ell_t$ components ($\mu_{t+1}=s\mu_t+\bar r\,\ell_t$), so the exact sensitivity has structural zeros in every $k_t$ slot — a useful check that the estimator finds them.

In [ ]:
reps_sens = 60
rows = []
for eta in lambdas:
    r = continuous_sensitivity_error(env, theta_star, eta=eta, n=cfg.n_aux, reps=reps_sens, generator=torch.Generator(device="cpu").manual_seed(0))
    rows.append({"eta": eta, "n": cfg.n_aux, "||bias||": r["total_bias_norm"].item(), "bias SE": r["bias_se"].sum().item(), "MSE": r["total_mse"].item()})
print(f"bias vs. eta at the training batch size n={cfg.n_aux}:")
display(pd.DataFrame(rows).set_index("eta"))

rows = []
for n in (cfg.n_aux, 4 * cfg.n_aux):
    r = continuous_sensitivity_error(env, theta_star, eta=REF_LAM, n=n, reps=reps_sens, generator=torch.Generator(device="cpu").manual_seed(0))
    rows.append({"n": n, "eta": REF_LAM, "||bias||": r["total_bias_norm"].item(), "variance": r["variance"].sum().item(), "MSE": r["total_mse"].item()})
print(f"\nvariance vs. n at fixed eta={REF_LAM}:")
display(pd.DataFrame(rows).set_index("n"))

exact_D = exact_coordinate_sensitivity(env, theta_star)
print(f"\nexact D_t: k-slots all zero? {bool((exact_D[..., 0] == 0).all())}; ||D_T||={exact_D[T].norm().item():.4f}")

### Mean-squared error vs the perturbation scale

Theorem "Bias and MSE of the gradient estimator" gives, at fixed budget,
$$\mathrm{MSE}(\hat g_{B,n,\lambda}) \lesssim \lambda^{2p} + \frac1{n\lambda^2} + \frac1{B\lambda^2}, \qquad p=2\ \text{here},$$
a U-shape in $\lambda$: perturbation bias dominates at large $\lambda$, estimator variance at small $\lambda$. On this benchmark the variance branch is the one that matters over the reference's whole grid, which is the quantitative reason training plateaus; the bias branch only takes over well beyond it.

In [ ]:
reps_mse = 40
lam_grid = [0.025, 0.05, 0.1, 0.2, 0.4, 0.8, 1.6]
mse_by_budget = {}
for scale in (1, 4):
    mse_by_budget[scale] = {
        lam: continuous_gradient_diagnostics(
            env, theta_star, lam=lam, B=scale * cfg.B, n_aux=scale * cfg.n_aux, reps=reps_mse,
            baseline=cfg.baseline, generator=torch.Generator(device="cpu").manual_seed(0),
        )["mse"].sum().item()
        for lam in lam_grid
    }

fig, ax = new_figure()
for i, (scale, mse) in enumerate(mse_by_budget.items()):
    viz.plot_horizon_scaling(mse, xlabel="perturbation scale λ", ylabel="squared gradient error vs ∇J^0(θ*)",
                             label=f"{scale}x training budget", color_index=i, integer_xaxis=False, ax=ax)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Gradient mean-squared error across perturbation scales")

### Perturbation coverage: $W_2(M_t^\lambda,\mu_t)\le\lambda|Z_t|\sqrt{1+\mu_t^2+\Sigma_t}$

The transport analogue of the discrete benchmarks' almost-sure $d_{TV}(M^\lambda,\mu)\le\lambda$ check, computed against the Gaussian chart the perturbation acts through. `within_bound` checks the bound on **every single draw**; `mean W2^2` is checked against its own closed form $\lambda^2\rho^2(\mu_t^2+1+\Sigma_t)$. With $\rho=1$ here (against LQ's $0.3$), the same $\lambda$ moves the population law about three times as far.

In [ ]:
coverage = {lam: continuous_perturbation_coverage(env, theta_star, lam=lam, n_samples=20_000,
                                                  generator=torch.Generator(device="cpu").manual_seed(0)) for lam in lambdas}
rows = []
for lam, res in coverage.items():
    for r in res:
        rows.append({"λ": lam, "t": r["t"], "mu_t": r["mu"].item(), "mean W2": r["mean_W2"].item(),
                     "max W2": r["max_W2"].item(), "mean W2^2": r["mean_W2_sq"].item(),
                     "predicted W2^2": r["predicted_mean_W2_sq"].item(), "within bound": r["within_bound"]})
df = pd.DataFrame(rows).set_index(["λ", "t"])
print(f"bound holds on every draw, at every (lambda, t): {df['within bound'].all()}")
df.loc[REF_LAM]

### Stability of the perturbed wealth marginal

The mean of the perturbed wealth marginal is exactly unaffected by $\lambda$ (the perturbation is centered, and $\mu_{t+1}=s\mu_t+\bar r\ell_t$ contains no $\lambda$), and the whole effect is a variance inflation $\Sigma_t^{\theta,\lambda}-\Sigma_t^{\theta,0}$ compounding forward. The simulated marginal is shown alongside both exact recursions as a check on the simulator.

In [ ]:
stability = {lam: continuous_state_marginal_stability(env, theta_star, lam=lam, n_samples=100_000,
                                                      generator=torch.Generator(device="cpu").manual_seed(0)) for lam in lambdas}

fig, ax = new_figure()
for i, lam in enumerate(lambdas):
    viz.plot_horizon_scaling({t: v for t, v in enumerate(stability[lam]["variance_inflation"].tolist())},
                             xlabel="t", ylabel="variance inflation: Σ_t^{θ,λ} - Σ_t^{θ,0}", label=f"λ={lam}", color_index=i, ax=ax)
ax.set_title("Perturbation-induced variance inflation over time")

s = stability[REF_LAM]
display(pd.DataFrame({
    "exact mean": s["exact_mean"].tolist(), "simulated mean": s["empirical_mean"].tolist(),
    "nominal variance": s["nominal_variance"].tolist(), "exact variance": s["exact_variance"].tolist(),
    "simulated variance": s["empirical_variance"].tolist(),
}, index=[f"t={t}" for t in range(T + 1)]))

## Summary

Total notebook runtime (including any training performed in this run):

In [ ]:
print(f"total notebook runtime: {time.perf_counter() - _notebook_start:.1f}s")